# Digital Wayfinder Content (S7)

Generates **content** for the digital wayfinders from evidence - never invented. Each panel
is assembled from the route geometry (heading + walk time to the gateway), the wayfinder
chain (next marker), the Gate 0C crossing caution, onward B-KQ destinations, and **real
transport context**: NaPTAN bus stops (Trust A) near each panel.

Two deployable artefacts:
- a flat **content pack** (CSV) for review, and
- a **panel JSON** with an explicit *static vs dynamic* schema, so live fields (bus
  departures, events, accessibility status) can be **updated** server-side at deployment.

Descriptive draft (content_version 0.1). Anchors provisional; bus stops are NaPTAN; no
funding claim. Wayfinder locations are from notebook 07.

In [ ]:
from __future__ import annotations
from pathlib import Path
import sys, warnings, json
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, geopandas as gpd, networkx as nx, osmnx as ox
import matplotlib.pyplot as plt
from shapely.geometry import Point

PROJECT_ROOT = Path.cwd()
PHASE1_ROOT = PROJECT_ROOT.parent if PROJECT_ROOT.name == "notebooks" else PROJECT_ROOT / "phase1_spinelens_ai"
sys.path.insert(0, str(PHASE1_ROOT / "src"))
from spinelens.spatial import audit
from spinelens.metrics import legibility as lg
from spinelens.models import wayfinding as wf
from spinelens.gate0b import utc_now_iso

DATA = PHASE1_ROOT / "data"
FIG_DIR = PHASE1_ROOT / "outputs" / "reports" / "wayfinder_content_media"
TABLES = PHASE1_ROOT / "outputs" / "tables"; EXPORTS = PHASE1_ROOT / "outputs" / "exports"; REPORTS = PHASE1_ROOT / "outputs" / "reports"
for d in (FIG_DIR, TABLES, EXPORTS, REPORTS): d.mkdir(parents=True, exist_ok=True)

G = ox.load_graphml(DATA / "raw" / "osm_network" / "gate0b_osm_walk_graph.graphml")
und = G.to_undirected(); largest = max(nx.connected_components(und), key=len)
coords = {n: (float(d["y"]), float(d["x"])) for n, d in G.nodes(data=True)}
cl = {n: c for n, c in coords.items() if n in largest}
import ast
def _hw(h): return h if isinstance(h, list) else (ast.literal_eval(h) if isinstance(h, str) and h.startswith("[") else [h])
node_severity = {n: 0.0 for n in G.nodes}
for u, v, d in G.edges(data=True):
    s = max((lg.crossing_severity(x) for x in _hw(d.get("highway"))), default=0.0)
    if s: node_severity[u] = max(node_severity[u], s); node_severity[v] = max(node_severity[v], s)

nodes = pd.read_csv(DATA / "route_nodes_phase1.csv").set_index("node_id")
def snap(nid): return audit.nearest_node(cl, (float(nodes.loc[nid,"latitude"]), float(nodes.loc[nid,"longitude"])))[0]
g_node = snap("ryder_street_pavilion_search_area")
GATEWAY = coords[g_node]
JUNCTION = (52.4864, -1.8844)

# Wayfinder locations from notebook 07
place = pd.read_csv(TABLES / "wayfinder_placement.csv").sort_values("rank").reset_index(drop=True)
# Active NaPTAN bus stops (Trust A)
bus = pd.read_csv(DATA / "raw" / "naptan" / "naptan_bus_stops_studyarea.csv")
if "Status" in bus: bus = bus[bus["Status"].fillna("active") == "active"]
print(f"wayfinders: {len(place)} | active NaPTAN bus stops: {len(bus)}")

## Generate content per wayfinder (evidence-traced)

In [ ]:
ONWARD = ["Aston University", "Millennium Point"]  # B-KQ destinations the gateway hands onward to
BUS_RADIUS_M = 120
ts = utc_now_iso()

def nearby_bus(lat, lon, r=BUS_RADIUS_M):
    d = bus.assign(d=bus.apply(lambda x: audit.haversine_m((lat, lon), (x.Latitude, x.Longitude)), axis=1))
    near = d[d.d <= r].sort_values("d")
    return list(dict.fromkeys(near["CommonName"].tolist()))  # dedupe, keep order

# distance to gateway for each wayfinder (for the marker chain)
place["dist_gw_m"] = place["node"].apply(lambda n: nx.shortest_path_length(G, n, g_node, weight="length"))
records = []
for i, row in place.iterrows():
    wid = f"W{int(row['rank'])}"
    lat, lon = float(row["lat"]), float(row["lon"])
    brg = lg.initial_bearing((lat, lon), GATEWAY)
    heading = wf.bearing_to_compass(brg)
    dist = float(row["dist_gw_m"]); wtime = wf.walk_time_minutes(dist)
    # next marker = nearest wayfinder closer to the gateway, else the gateway
    closer = place[place["dist_gw_m"] < dist]
    if len(closer):
        nxt = closer.iloc[(closer["lat"].sub(lat).pow(2) + closer["lon"].sub(lon).pow(2)).values.argmin()]
        next_marker = f"W{int(nxt['rank'])}"
    else:
        next_marker = "Ryder Street gateway"
    stops = nearby_bus(lat, lon)
    sev = node_severity.get(int(row["node"]), 0.0)
    near_junction = audit.haversine_m((lat, lon), JUNCTION) <= 200
    caution = ("Major road crossing ahead - use the signalised crossing" if (sev >= 2 or near_junction) else "")
    approaches = row["routes_served"]
    directional = f"B-KQ {heading} - Ryder Street gateway, {wtime} min. Continue to {next_marker}."
    records.append({
        "wayfinder_id": wid, "intervention_type": row["intervention_type"],
        "approaches_served": approaches, "lat": round(lat, 6), "lon": round(lon, 6),
        "heading_to_gateway": heading, "walk_time_to_gateway_min": wtime,
        "next_marker": next_marker, "directional_text": directional,
        "nearby_bus_stops": "; ".join(stops[:3]), "bus_stop_count": len(stops),
        "onward_destinations": ", ".join(ONWARD),
        "crossing_caution": caution,
        "accessibility_note": "Step-free route assumed - confirm on site",
        "confidence_level": "provisional (anchors not field-validated)",
        "content_version": "0.1-draft", "last_updated_utc": ts,
        "qr_target": f"https://bkq.example/wayfinder/{wid.lower()}",
    })
content = pd.DataFrame(records)
content.to_csv(TABLES / "wayfinder_content_pack.csv", index=False)
display(content[["wayfinder_id", "approaches_served", "heading_to_gateway",
                 "walk_time_to_gateway_min", "next_marker", "nearby_bus_stops", "crossing_caution"]])

In [ ]:
# --- Nechells-side wayfinder content (funnel to the barrier + onward into the cluster) ---
# These wayfinders point toward the barrier / INTO the cluster (not back to the city-core
# gateway) and have no pavilion, so they carry hub-lite content. Same evidence basis
# (OSM network + Gate 0C + NaPTAN).
onward = pd.read_csv(TABLES / "wayfinder_placement_onward.csv")
CLUSTER_NAME = {"aston_university": "Aston University", "millennium_point": "Millennium Point",
                "bcu_parkside": "BCU City Centre", "steamhouse": "STEAMhouse",
                "innovation_birmingham": "Innovation Birmingham"}
CLUSTER_NODE = {d: snap(d) for d in CLUSTER_NAME}

onward_records = []
for _, row in onward.iterrows():
    wid = f"W{int(row['rank'])}"
    lat, lon = float(row["lat"]), float(row["lon"])
    node = int(row["node"])
    served = [s for s in str(row["serves_destinations"]).split(",") if s]
    # nearest served cluster destination drives heading + onward walk time
    dists = {d: nx.shortest_path_length(G, node, CLUSTER_NODE[d], weight="length") for d in served}
    primary = min(dists, key=dists.get)
    wtime = wf.walk_time_minutes(dists[primary])
    heading = wf.bearing_to_compass(lg.initial_bearing((lat, lon), coords[CLUSTER_NODE[primary]]))
    stops = nearby_bus(lat, lon)
    sev = node_severity.get(node, 0.0)
    near_junction = audit.haversine_m((lat, lon), JUNCTION) <= 200
    caution = ("Major road crossing ahead - use the signalised crossing" if (sev >= 2 or near_junction) else "")
    names = ", ".join(CLUSTER_NAME[d] for d in served)
    onward_records.append({
        "wayfinder_id": wid, "intervention_type": row["intervention_type"],
        "approaches_served": "bkq_cluster", "lat": round(lat, 6), "lon": round(lon, 6),
        "heading_to_gateway": heading, "walk_time_to_gateway_min": wtime,
        "next_marker": CLUSTER_NAME[primary],
        "directional_text": f"B-KQ cluster {heading}: {names} - {wtime} min",
        "nearby_bus_stops": "; ".join(stops[:3]), "bus_stop_count": len(stops),
        "onward_destinations": names,
        "crossing_caution": caution,
        "accessibility_note": "Step-free route assumed - confirm on site",
        "confidence_level": "provisional (cluster anchors not field-validated)",
        "content_version": "0.1-draft", "last_updated_utc": ts,
        "qr_target": f"https://bkq.example/wayfinder/{wid.lower()}",
        "leg": row.get("leg", "onward_cluster"), "serves_destinations": ",".join(served),
    })
content = pd.concat([content, pd.DataFrame(onward_records)], ignore_index=True)
content["leg"] = content["leg"].fillna("inbound_gateway")
content.to_csv(TABLES / "wayfinder_content_pack.csv", index=False)
print(f"appended {len(onward_records)} Nechells-side wayfinders -> total {len(content)}")
display(pd.DataFrame(onward_records)[["wayfinder_id", "leg", "intervention_type", "onward_destinations",
                                      "heading_to_gateway", "walk_time_to_gateway_min", "crossing_caution"]])

## Per-approach origin panels

In [ ]:
ORIGINS = {"colmore_row": "Colmore Row", "new_street_station": "New Street Station",
           "moor_street_queensway": "Moor Street", "snow_hill_station": "Snow Hill Station",
           "dartmouth_middleway_nechells": "Nechells / Dartmouth"}
origin_panels = []
for oid, name in ORIGINS.items():
    o = snap(oid); olat, olon = coords[o]
    try:
        dist = nx.shortest_path_length(G, o, g_node, weight="length")
    except nx.NetworkXNoPath:
        dist = audit.haversine_m(coords[o], GATEWAY)
    heading = wf.bearing_to_compass(lg.initial_bearing((olat, olon), GATEWAY))
    stops = nearby_bus(olat, olon, r=150)
    origin_panels.append({
        "origin": name, "heading_to_gateway": heading,
        "walk_time_to_gateway_min": wf.walk_time_minutes(dist),
        "welcome_text": f"Welcome - B-KQ is {wf.walk_time_minutes(dist)} min {heading}, "
                        f"follow the amber route to the Ryder Street gateway.",
        "nearby_bus_stops": "; ".join(stops[:3]), "bus_stop_count": len(stops),
    })
origin_df = pd.DataFrame(origin_panels)
display(origin_df)

## Deployable panel JSON (static vs dynamic schema)

In [ ]:
panel_json = {
    "schema_version": "0.1",
    "generated_utc": ts,
    "notes": "Phase 1 draft content. Static fields are evidence-derived; dynamic fields are "
             "populated by live services at deployment and are UPDATEABLE.",
    "static_fields": ["heading_to_gateway", "walk_time_to_gateway_min", "next_marker",
                      "onward_destinations", "nearby_bus_stops", "crossing_caution", "qr_target"],
    "dynamic_fields": ["live_bus_departures", "event_alerts", "accessibility_status", "footfall_now"],
    "data_sources": {"network": "OSM (Level 3)", "bus_stops": "NaPTAN (Trust A)",
                     "crossing": "DfT AADF + STATS19 (Gate 0C)"},
    "wayfinders": json.loads(content.to_json(orient="records")),
    "origin_panels": json.loads(origin_df.to_json(orient="records")),
}
(EXPORTS / "wayfinder_panel_content.json").write_text(json.dumps(panel_json, indent=2), encoding="utf-8")
print("exported wayfinder_panel_content.json with",
      len(panel_json["wayfinders"]), "wayfinders and", len(panel_json["origin_panels"]), "origin panels")
print("dynamic (updateable) fields:", panel_json["dynamic_fields"])

## Visuals

In [ ]:
base = ox.graph_to_gdfs(G, nodes=False).to_crs(27700)
def to_m(lat, lon):
    return gpd.GeoSeries([Point(lon, lat)], crs=4326).to_crs(27700).iloc[0]
gm = to_m(*GATEWAY)
wf_m = [to_m(r.lat, r.lon) for r in content.itertuples()]
bus_m = gpd.GeoSeries(gpd.points_from_xy(bus.Longitude, bus.Latitude, crs=4326)).to_crs(27700)
minx = min(p.x for p in wf_m) - 250; maxx = max(p.x for p in wf_m) + 250
miny = min(p.y for p in wf_m) - 250; maxy = max(p.y for p in wf_m) + 250

fig, ax = plt.subplots(figsize=(11, 9))
base.cx[minx:maxx, miny:maxy].plot(ax=ax, color="#ececec", linewidth=0.4, zorder=1)
bm = bus_m[(bus_m.x.between(minx, maxx)) & (bus_m.y.between(miny, maxy))]
ax.scatter(bm.x, bm.y, s=18, color="#34a853", zorder=3, label=f"NaPTAN bus stops ({len(bm)})")
for p, r in zip(wf_m, content.itertuples()):
    ax.scatter(p.x, p.y, marker="*", s=300, color="#f59e0b", edgecolor="black", zorder=5)
    ax.annotate(r.wayfinder_id, (p.x, p.y), fontsize=9, xytext=(5, 5), textcoords="offset points")
ax.scatter(gm.x, gm.y, marker="s", s=150, color="black", edgecolor="white", zorder=6, label="Ryder gateway")
ax.set_xlim(minx, maxx); ax.set_ylim(miny, maxy); ax.set_aspect("equal"); ax.legend(loc="upper left")
ax.set_title("Wayfinders (amber) with NaPTAN bus-stop context")
fig.savefig(FIG_DIR / "figQ_wayfinder_bus_context.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
# Illustrative panel mock for the top wayfinder
w = content.iloc[0]
fig, ax = plt.subplots(figsize=(5.5, 8)); ax.axis("off")
ax.add_patch(plt.Rectangle((0, 0), 1, 1, color="#0f172a"))
ax.text(0.5, 0.93, "B-KQ", color="#f59e0b", fontsize=30, ha="center", weight="bold")
ax.text(0.5, 0.86, "Innovation Spine", color="#f8fafc", fontsize=12, ha="center")
ax.text(0.08, 0.74, f"{w['heading_to_gateway']}   Ryder Street gateway", color="#f8fafc", fontsize=15, weight="bold")
ax.text(0.08, 0.68, f"{w['walk_time_to_gateway_min']} min walk", color="#34d399", fontsize=14)
ax.text(0.08, 0.60, "Onward in B-KQ:", color="#94a3b8", fontsize=11)
ax.text(0.08, 0.56, w["onward_destinations"], color="#f8fafc", fontsize=12)
ax.text(0.08, 0.48, "Buses nearby:", color="#94a3b8", fontsize=11)
ax.text(0.08, 0.44, (w["nearby_bus_stops"] or "none within 120 m"), color="#f8fafc", fontsize=11, wrap=True)
if w["crossing_caution"]:
    ax.text(0.08, 0.34, "! " + w["crossing_caution"], color="#fbbf24", fontsize=11, wrap=True)
ax.text(0.08, 0.16, f"Continue to {w['next_marker']}", color="#f8fafc", fontsize=12)
ax.text(0.08, 0.06, f"v{w['content_version']} - scan for live map + buses", color="#64748b", fontsize=9)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title(f"Illustrative panel - {w['wayfinder_id']} (content auto-generated)")
fig.savefig(FIG_DIR / "figR_panel_mock.png", dpi=130, bbox_inches="tight"); plt.show()

## Note

In [ ]:
lines = [
    "# Digital Wayfinder Content Note (S7)",
    "", f"Generated: {ts}. content_version 0.1-draft. Descriptive; no funding claim.",
    "", "## Approach",
    "",
    "Every field is evidence-derived, not invented:",
    "- heading + walk time: OSM network geometry (Level 3).",
    "- next marker: the wayfinder chain toward the gateway.",
    "- crossing caution: Gate 0C (DfT AADF + STATS19).",
    "- bus stops: NaPTAN (Trust A), active stops within 120 m.",
    "- onward: the gateway/pavilion hands people onward to B-KQ destinations.",
    "",
    "## Updateability",
    "",
    "The panel JSON separates static (evidence-derived) from dynamic fields",
    "(live_bus_departures, event_alerts, accessibility_status, footfall_now) that live",
    "services populate at deployment - so content stays current without re-authoring.",
    "",
    "## Coverage",
    "",
    f"- Wayfinders with content: {len(content)}.",
    f"- Wayfinders with a bus stop within 120 m: {(content['bus_stop_count']>0).sum()}.",
    f"- Wayfinders carrying a crossing caution: {(content['crossing_caution']!='').sum()}.",
    f"- Origin welcome panels: {len(origin_df)} (one per approach).",
    "",
    "## Caveats",
    "",
    "- Anchors are provisional; accessibility is assumed step-free pending on-site audit.",
    "- Bus-stop presence is from NaPTAN; live departures require a feed (e.g. BODS) at deployment.",
    "- Content tone/branding is indicative, for design review.",
]
(REPORTS / "wayfinder_content_note.md").write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines[:16]))

## What this unlocks

Deployment-ready, evidence-traced wayfinder content for every approach and panel, with a
clear static/dynamic split so it is genuinely updateable, and real NaPTAN transport context
so it is defensible. Feeds the budget pack (S8) and a future React/panel front end.

# Engagement layer: tiers, dwell modules, civic sponsorship

For people who **stop** to engage. The system is **tiered** to keep cost realistic:
Tier 1 = the gateway pavilion hub (full interactive), Tier 2 = a few interactive totems at
convergence/origin nodes, Tier 3 = low-cost glanceable markers + QR-to-phone.

Priority is **pedestrian movement to B-KQ** - every dwell module is orientation/walking
focused. Bus stops remain only light interchange context (where bus users become
pedestrians), never a feature; there is no live-transport module.

Dwell modules enabled: interactive map + find-by-need, what's on, accessibility/language/QR.
Sponsorship: **civic partner** (bounded, non-intrusive, part-funds the free service).

In [ ]:
ENABLED_MODULES = {"directory_find_by_need", "interactive_map", "whats_on",
                   "accessibility_language", "qr_handoff"}
DWELL_MODULES = {
    "interactive_map": "Pan/zoom B-KQ map, 'you are here', tap-to-route incl. step-free (pedestrian routing)",
    "directory_find_by_need": "Choose by need (study/work/eat/health/innovation) -> walking directions",
    "whats_on": "B-KQ events, open days, exhibitions, markets (updateable)",
    "accessibility_language": "Text size, audio, language toggle, step-free mode",
    "qr_handoff": "Scan to send the pedestrian map + route to your phone",
    "gateway_teaser": "A small B-KQ teaser as you approach the gateway (universal, light)",
}
SPONSORSHIP_POLICY = {
    "model": "civic_partner",
    "principle": "Curated B-KQ tenants/universities/local independents; part-funds the free public wayfinding service.",
    "guardrails": ["bounded non-intrusive slot (<=15% screen)", "relevant + contextual",
                   "no tracking", "never blocks wayfinding or accessibility", "screened tiers only (hub/totem)"],
}

# Feature allocation by route: city-core markers stay light (the pavilion is their hub);
# Nechells markers carry hub-lite content (no pavilion); every panel gets a gateway teaser.
content["approaches_list"] = content["approaches_served"].apply(lambda s: [x for x in str(s).split(",") if x])
content["tier"] = [wf.assign_tier(len(a), it) for a, it in zip(content["approaches_list"], content["intervention_type"])]
content["content_role"] = [wf.content_role(t, a) for t, a in zip(content["tier"], content["approaches_list"])]
content["dwell_modules"] = [";".join(wf.effective_modules(t, ENABLED_MODULES, a))
                            for t, a in zip(content["tier"], content["approaches_list"])]
content["sponsorship_enabled"] = [wf.sponsorship_slot(t)["enabled"] for t in content["tier"]]
content.drop(columns=["approaches_list"]).to_csv(TABLES / "wayfinder_content_pack.csv", index=False)

tier_counts = content["tier"].value_counts().to_dict()
role_counts = content["content_role"].value_counts().to_dict()
print("Tier split:", tier_counts, "| content roles:", role_counts)
print("Nechells wayfinders carry hub-lite content (no pavilion); all panels get a gateway teaser.")
display(content[["wayfinder_id", "approaches_served", "tier", "content_role", "dwell_modules", "sponsorship_enabled"]])

In [ ]:
hub_panel = {
    "id": "HUB-pavilion", "tier": wf.TIER_HUB, "content_role": "hub",
    "location": {"lat": GATEWAY[0], "lon": GATEWAY[1]},
    "role": "Gateway pavilion: public, B-KQ-wide information hub and civic living room",
    "modules": wf.modules_for_tier(wf.TIER_HUB, ENABLED_MODULES),
    "find_by_need": ["study", "work", "eat", "health", "innovation"],
    "onward_destinations": ONWARD,
    "sponsorship": wf.sponsorship_slot(wf.TIER_HUB),
}
for r in panel_json["wayfinders"]:
    row = content.loc[content["wayfinder_id"] == r["wayfinder_id"]].iloc[0]
    r["tier"] = row["tier"]; r["content_role"] = row["content_role"]
    r["dwell_modules"] = row["dwell_modules"].split(";") if row["dwell_modules"] else []
    r["sponsorship"] = wf.sponsorship_slot(row["tier"])
panel_json["hub_panel"] = hub_panel
panel_json["dwell_modules_catalogue"] = DWELL_MODULES
panel_json["sponsorship_policy"] = SPONSORSHIP_POLICY
panel_json["priority"] = "pedestrian_movement_to_bkq"
panel_json["feature_allocation"] = {
    "city_core_routes": "light markers + QR + gateway teaser; full dwell at the pavilion hub",
    "nechells_dartmouth": "hub-lite dwell content on the wayfinders (no pavilion)",
    "all_routes": "small gateway/B-KQ teaser on approach",
}
(EXPORTS / "wayfinder_panel_content.json").write_text(json.dumps(panel_json, indent=2), encoding="utf-8")
print("panel JSON updated: per-panel content roles, Nechells hub-lite boost, universal teaser.")
print("dwell_no_pavilion panels:", [r["wayfinder_id"] for r in panel_json["wayfinders"] if r["content_role"] == "dwell_no_pavilion"])

In [ ]:
# Tier-1 hub panel mock (illustrative; pedestrian-first dwell layout)
fig, ax = plt.subplots(figsize=(7, 9)); ax.axis("off")
ax.add_patch(plt.Rectangle((0, 0), 1, 1, color="#0f172a"))
ax.text(0.5, 0.95, "B-KQ GATEWAY", color="#f59e0b", fontsize=24, ha="center", weight="bold")
ax.text(0.5, 0.905, "find your way into the Knowledge Quarter", color="#cbd5e1", fontsize=11, ha="center")
panels = [
    ("MAP", "you are here - tap a place to walk to (step-free option)", "#1e293b"),
    ("FIND BY NEED", "study | work | eat | health | innovation", "#1e293b"),
    ("WHAT'S ON", "open days | exhibitions | STEAMhouse | markets", "#1e293b"),
    ("ACCESS", "text size | audio | language | step-free", "#1e293b"),
]
y = 0.80
for title, body, col in panels:
    ax.add_patch(plt.Rectangle((0.06, y-0.12), 0.88, 0.11, color=col))
    ax.text(0.09, y-0.03, title, color="#f59e0b", fontsize=13, weight="bold")
    ax.text(0.09, y-0.085, body, color="#e2e8f0", fontsize=10)
    y -= 0.15
ax.text(0.09, 0.16, "Scan to take the map + route with you", color="#34d399", fontsize=11)
ax.add_patch(plt.Rectangle((0.06, 0.03, ), 0.88, 0.06, color="#11212f"))
ax.text(0.09, 0.055, "Supported by a B-KQ civic partner", color="#94a3b8", fontsize=9, style="italic")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title("Illustrative Tier-1 hub (pavilion) - dwell layout")
fig.savefig(FIG_DIR / "figS_hub_dwell_mock.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
lines = [
    "# Wayfinder Engagement Layer Note (S7+)",
    "", f"Generated: {ts}. Pedestrian-first. Descriptive draft; no funding claim.",
    "", "## Feature allocation by route",
    "",
    "- **City-core routes (Colmore, New St, Moor St, Snow Hill)** reach the pavilion hub, so their",
    "  en-route markers stay light (glance + QR + gateway teaser); full dwell is at the pavilion.",
    "- **Nechells / Dartmouth** has NO pavilion, so its wayfinders carry hub-lite dwell content",
    "  (map + find-by-need + what's on) - they substitute for the missing hub.",
    "- **All routes** get a small gateway/B-KQ teaser on approach.",
    "", "## Tiered system (cost-realistic)",
    "",
    f"- Tier 1 hub (pavilion): 1.",
    f"- Tier 2 totems (interactive): {tier_counts.get(wf.TIER_TOTEM, 0)}.",
    f"- Tier 3 markers (glance + QR): {tier_counts.get(wf.TIER_MARKER, 0)}.",
    f"- Content roles: {role_counts}.",
    "", "## Dwell modules (orientation / pedestrian focused)", "",
]
for k, v in DWELL_MODULES.items():
    lines.append(f"- **{k}**: {v}")
lines += [
    "", "## Civic-partner sponsorship", "",
    SPONSORSHIP_POLICY["principle"], "",
] + [f"- {g}" for g in SPONSORSHIP_POLICY["guardrails"]] + [
    "",
    "Revenue from sponsorship part-funds the free public service (an investibility lever),",
    "without compromising wayfinding, accessibility or the civic feel.",
    "", "## Priority", "",
    "Pedestrian movement to B-KQ is the priority. Bus stops are light interchange context only;",
    "there is no live-transport module.",
    "", "## Caveats", "",
    "- Modules, tiering and sponsorship are a design proposal for review, not a built system.",
    "- Tier/role counts feed the budget pack (S8); event feeds are deployment integrations.",
]
(REPORTS / "wayfinder_engagement_note.md").write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines[:14]))

## What this unlocks

A realistic, tiered wayfinder experience that serves both moving and dwelling pedestrians,
with evidence-traced content, an updateable schema, and a civic-partner sponsorship model
that part-funds the free service. Tier counts feed the budget pack (S8).